In [3]:
import json, time
from datetime import date 
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from reader.reader import answer, load_prompt, MODEL_REPO, MODEL_REVISION, MAX_NEW_TOKENS
from eval.evaluation import write_predictions, score_answers


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()

SPLITS_DIR = ROOT / "splits"
GOLD_DIR = ROOT / "results" / "gold"
RUNS_DIR = ROOT / "data" / "runs"
RESULTS_RUNS_DIR = ROOT / "results" / "runs"
CORPUS_PATH = ROOT / "data" / "corpus" / "pooled_corpus.parquet"

SLICE = "pilot"
TOP_K = 10
prompt = load_prompt("v1")

with open(SPLITS_DIR / f"{SLICE}.json") as f:
    slice_ids = json.load(f)
with open(GOLD_DIR / f"{SLICE}_gold.json") as f:
    gold = json.load(f)


# question text + the distractor context (context is only used for the distractor/oracle reader tests)
slice_q = (load_dataset("hotpotqa/hotpot_qa", "distractor")["train"]
            .select_columns(["id", "question", "type", "context"])
            .to_pandas().set_index("id").loc[slice_ids])
qid_to_type = slice_q["type"].to_dict()
gold_titles = {g["_id"]: {t for t, _ in g["supporting_facts"]} for g in gold}

# pid -> paragraph, for retrieval-based contexts
corpus = pd.read_parquet(CORPUS_PATH, columns=["pid", "title", "text"]).set_index("pid")
with open(RUNS_DIR / f"rrf_{SLICE}.json") as f:
    rrf_run = json.load(f)
    
def distractor_paragraphs(qid):
    """The 10 paragraphs given with the question (2 gold + 8 distractors), in their given order."""
    context = slice_q.at[qid, "context"]
    return [{"title": t, "text": "".join(s)} for t, s in zip(context["title"], context["sentences"])]
    
def oracle_paragraphs(qid):
    """Only the 2 gold paragraphs (upper bound for the reader; uses labels, train slices only)."""
    return [p for p in distractor_paragraphs(qid) if p["title"] in gold_titles[qid]]

def rrf_paragraphs(qid, k=TOP_K):
    """Top-k paragraphs of our hop-1 retriever (RRF of BM25 + Contriever)."""
    return [{"title": corpus.at[pid, "title"], "text": corpus.at[pid, "text"]} for pid in rrf_run[qid][:k]]


SETTINGS = {
    "closed_book": lambda qid: None,
    "oracle": oracle_paragraphs,
    "distractor": distractor_paragraphs,
    "rrf_top10": rrf_paragraphs,
}

In [4]:
def run_reader(setting, prompt=prompt):
    """Answer every slice question with one context setting; save predictions, per-question scores, metrics and config."""
    get_paragraphs = SETTINGS[setting]
    run_id = f"{date.today():%Y%m%d}_{SLICE}_{setting}_{prompt['version']}"
    run_dir = RESULTS_RUNS_DIR / run_id

    answers, seconds = {}, []
    for qid in slice_ids:
        start = time.time()
        answers[qid] = answer(slice_q.at[qid, "question"], get_paragraphs(qid), prompt)
        seconds.append(time.time() - start)

    predictions = write_predictions(answers, slice_ids, run_dir / "predictions.json")
    per_question, summary = score_answers(predictions, gold, qid_to_type)
    per_question.to_csv(run_dir / "per_question.csv", index=False)

    summary["seconds_per_question"] = sum(seconds) / len(seconds)
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(summary, f, indent=2)
    with open(run_dir / "config.json", "w") as f:
        json.dump({"run_id": run_id, "slice": SLICE, "n": len(slice_ids), "setting": setting,
                    "top_k": TOP_K if setting == "rrf_top10" else None,
                    "model": MODEL_REPO, "revision": MODEL_REVISION,
                    "prompt_version": prompt["version"], "prompt_sha1": prompt["sha1"],
                    "decoding": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS}}, f, indent=2)

    print(f"{setting:12s} EM {summary['overall']['em']:5.1f}  F1 {summary['overall']['f1']:5.1f}  "
        f"(bridge {summary['bridge']['em']:.1f} / comparison {summary['comparison']['em']:.1f} EM)  "
        f"{summary['seconds_per_question']:.2f} s/q  -> {run_dir.name}")
    return per_question, summary


In [5]:
results = {}
for setting in ["distractor", "oracle", "closed_book", "rrf_top10"]:
    results[setting] = run_reader(setting)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

distractor   EM  51.0  F1  65.3  (bridge 49.4 / comparison 57.5 EM)  0.94 s/q  -> 20261126_pilot_distractor_v1
oracle       EM  57.5  F1  73.4  (bridge 58.8 / comparison 52.5 EM)  0.40 s/q  -> 20261126_pilot_oracle_v1
closed_book  EM  20.0  F1  27.8  (bridge 11.2 / comparison 55.0 EM)  0.23 s/q  -> 20261126_pilot_closed_book_v1
rrf_top10    EM  36.0  F1  46.3  (bridge 30.6 / comparison 57.5 EM)  0.84 s/q  -> 20261126_pilot_rrf_top10_v1
